# Test Scala Data Provenance

### Prerequisites

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

### Import libraries

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension

In [4]:
version

1 deprecation (since 2.13.3); re-run enabling -deprecation for details, or try -help


res4: org.apache.spark.sql.Column = version()

### Initialize spark session

In [5]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .withExtensions(new ProvenanceExtension())
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.ProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()


println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")
  

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

// spark.experimental.extraOptimizations = Seq(LogicalPlanWithProvenance(spark))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 14:51:27 INFO SparkContext: Running Spark version 4.1.1
26/05/12 14:51:27 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/05/12 14:51:27 INFO SparkContext: Java version 17.0.10+7
26/05/12 14:51:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/12 14:51:27 INFO ResourceUtils: ==============================================================
26/05/12 14:51:27 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/12 14:51:27 INFO ResourceUtils: ==============================================================
26/05/12 14:51:27 INFO SparkContext: Submitted application: notebook
26/05/12 14:51:27 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/05/12 14:51:27 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/05/12 14:51:27 INFO SecurityManager: Changing view acls groups to

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@6f458c9c

### Create spark dataframe

We create a dataframe with duplicates to test the provenance annotation.
 
The provenance annotation will count the number of duplicates for each tuple.

In [ ]:
spark.conf.set("spark.provenance.enabled", "true")

val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")//.addProvenanceColumn

// df.printSchema()

// df.show()

org.apache.spark.sql.catalyst.analysis.UnresolvedException: [INTERNAL_ERROR] Invalid call to toAttribute on unresolved object SQLSTATE: XX000

In [ ]:
df.queryExecution.analyzed.transformUp {
    x => x match {
    case p: Project => println(s"Project: ${p.projectList}"); x
    case _: Any => println("toto"); x
    }}

toto
Project: List(_1#13 AS A#16, _2#14 AS B#17, _3#15 AS C#18)


res17: LogicalPlan = Project(
  projectList = List(
    Alias(
      child = AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      name = "A"
    ),
    Alias(
      child = AttributeReference(
        name = "_2",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      name = "B"
    ),
    Alias(
      child = AttributeReference(
        name = "_3",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      name = "C"
    )
  ),
  child = LocalRelation(
    output = List(
      AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
...

In [28]:
// val dfWithProvenance: DataFrame = df
//     .groupBy("A", "B", "C")
//     .count()
//     .withColumnRenamed("count", "provenance")
//     .orderBy("A", "B", "C")
    
// dfWithProvenance.show()

val dfWithProvenance : DataFrame = df.addProvenanceColumn
dfWithProvenance.show()


+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|              0|
|  a|  b|  c|              1|
|  d|  b|  e|              2|
|  d|  b|  e|              3|
|  d|  b|  e|              4|
|  d|  b|  e|              5|
|  d|  b|  e|              6|
|  f|  g|  e|              7|
+---+---+---+---------------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [ ]:
val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C", "_provenance_tag")
    .orderBy("A", "B", "C")

df2WithProvenance.show()


org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `_provenance_tag` cannot be resolved. Did you mean one of the following? [`A`, `B`, `C`]. SQLSTATE: 42703;
'Project [A#284, B#285, C#316, '_provenance_tag]
+- Project [B#285, A#284, C#316]
   +- Join Inner, (B#285 = B#315)
      :- Project [A#284, B#285]
      :  +- Project [A#284, B#285, C#286, monotonically_increasing_id() AS _provenance_tag#297L]
      :     +- Project [_1#281 AS A#284, _2#282 AS B#285, _3#283 AS C#286]
      :        +- LocalRelation [_1#281, _2#282, _3#283]
      +- Project [B#315, C#316]
         +- Project [A#314, B#315, C#316, monotonically_increasing_id() AS _provenance_tag#317L]
            +- Project [_1#311 AS A#314, _2#312 AS B#315, _3#313 AS C#316]
               +- LocalRelation [_1#311, _2#312, _3#313]
